In [1]:
import time
import warnings

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import norm

from model import TransferModel
from config import SlimConfig
from data_processing import DataProcessor
from real_score import load_line, set_line_context, score_line   # your copy with the settings loop

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- settings -------------------------------------------------------------
K, LAM, ALPHA = 20, 100, 1.5  # model config you'd submit (see validate_loco.ipynb)
CONTEXT = "A"                  # which context's controls to test against
N_TARGETS = 30                 # sampled challenge targets (more = slower, steadier numbers)
SEED = 1
FDR = 0.05                     # BH threshold for "significant"

In [2]:
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [3]:
gene_names = pd.read_csv("../../data/controls/gene_names.csv")["gene_name"].tolist()
dp = DataProcessor(gene_names)
kd = 0.9

In [4]:
train_names = list(DATASETS)                                   # all four training lines

model = TransferModel(gene_names, '../../data/embeddings/gene_string_embeddings.v0.3.h5', SlimConfig())
model.config.K, model.config.lambda_reg = K, LAM

measured_genes = {name: dp.load_dataset(name).var_names for name in train_names}
model.fit(*dp.build_Y(train_names, genes="union"), measured_genes)

  Fitted on 15627 perts, STRING coverage: 15118/15627
  direct transfer from ('K562',): 9675 targets have a measurement


In [5]:
xaira_data = load_line('HCT116', gene_names)
vcc2025_data = load_line('H1', gene_names, target_umi = 20000)

lines_dict = {
    'HCT116': xaira_data,
    'H1': vcc2025_data
}

HCT116: 18400 controls, 198 targets (cells per target median 184, min 100), 100 dropped (7s)
H1: 18400 controls, 97 targets (cells per target median 400, min 139), 3 dropped, control UMI/cell median 54616 -> 20009 (p=0.366) (49s)


In [6]:
GRID = {}

for N in (25, 50, 100):
    for top_gain in (2.5, 4):
        for base_gain in (0.5, 1):
            GRID[f'{N}_{top_gain}_{base_gain}'] = {
                "gain": base_gain,
                "n_call": N,
                "gain_top": top_gain
            }
GRID[f'{N}_2.5_2.5'] = {
    "gain": 2.5,
    "n_call": N,
    "gain_top": 2.5
}

GRID

{'25_2.5_0.5': {'gain': 0.5, 'n_call': 25, 'gain_top': 2.5},
 '25_2.5_1': {'gain': 1, 'n_call': 25, 'gain_top': 2.5},
 '25_4_0.5': {'gain': 0.5, 'n_call': 25, 'gain_top': 4},
 '25_4_1': {'gain': 1, 'n_call': 25, 'gain_top': 4},
 '50_2.5_0.5': {'gain': 0.5, 'n_call': 50, 'gain_top': 2.5},
 '50_2.5_1': {'gain': 1, 'n_call': 50, 'gain_top': 2.5},
 '50_4_0.5': {'gain': 0.5, 'n_call': 50, 'gain_top': 4},
 '50_4_1': {'gain': 1, 'n_call': 50, 'gain_top': 4},
 '100_2.5_0.5': {'gain': 0.5, 'n_call': 100, 'gain_top': 2.5},
 '100_2.5_1': {'gain': 1, 'n_call': 100, 'gain_top': 2.5},
 '100_4_0.5': {'gain': 0.5, 'n_call': 100, 'gain_top': 4},
 '100_4_1': {'gain': 1, 'n_call': 100, 'gain_top': 4},
 '100_2.5_2.5': {'gain': 2.5, 'n_call': 100, 'gain_top': 2.5}}

In [7]:
results = {}

for name, line in lines_dict.items():
    set_line_context(model, line, dp, train_names)
    scaled_score, raw_score = score_line(model, line, GRID, kd=kd, alpha=ALPHA, seed=SEED)
    results[name] = [scaled_score, raw_score]

results

HCT116 {'K562': np.float64(0.27), 'RPE1': np.float64(0.3), 'HepG2': np.float64(0.19), 'Jurkat': np.float64(0.24)}
scorer: 10473 DE-tested genes, 18400 controls (6s)
reference + replicate: 125s, median real DE calls 3
25_2.5_0.5: 122s
25_2.5_1: 121s
25_4_0.5: 125s
25_4_1: 127s
50_2.5_0.5: 140s
50_2.5_1: 122s
50_4_0.5: 130s
50_4_1: 128s
100_2.5_0.5: 138s
100_2.5_1: 122s
100_4_0.5: 121s
100_4_1: 129s
100_2.5_2.5: 122s
H1 {'K562': np.float64(0.28), 'RPE1': np.float64(0.23), 'HepG2': np.float64(0.23), 'Jurkat': np.float64(0.25)}
scorer: 10776 DE-tested genes, 18400 controls (12s)
reference + replicate: 130s, median real DE calls 270
25_2.5_0.5: 71s
25_2.5_1: 70s
25_4_0.5: 70s
25_4_1: 71s
50_2.5_0.5: 71s
50_2.5_1: 71s
50_4_0.5: 71s
50_4_1: 72s
100_2.5_0.5: 71s
100_2.5_1: 71s
100_4_0.5: 71s
100_4_1: 71s
100_2.5_2.5: 70s


{'HCT116': [                     pds       mse       fid     reach       jac      nmae  \
  baseline        0.000000  0.000000  0.000000  0.000000  0.000000 -0.000000   
  replicate       1.000000  1.000000  1.000000  1.000000  1.000000  1.000000   
  25_2.5_0.5      0.315647  0.209666  0.152423  0.073310  0.021486  0.051395   
  25_2.5_1        0.331064  0.334664  0.136165  0.196648  0.015822  0.083094   
  25_4_0.5        0.321300  0.772504  0.153634  0.056388  0.022182  0.015046   
  25_4_1          0.135534  0.933089  0.086195  0.128062  0.016085  0.050829   
  50_2.5_0.5      0.220195  0.302585  0.136453  0.096686  0.017019  0.054505   
  50_2.5_1        0.251799  0.413995  0.122169  0.115991  0.014291  0.063227   
  50_4_0.5        0.334918  0.940205  0.119764  0.140670  0.017798  0.021866   
  50_4_1          0.108813  1.000000  0.105824  0.169754  0.013979  0.059503   
  100_2.5_0.5     0.260534  0.417295  0.103828  0.145482  0.012664  0.080927   
  100_2.5_1       0.188592  0.

In [8]:
results['H1'][1]

,pds,mse,fid,reach,jac,nmae,n_sig
baseline,0.500000,0.958439,0.550502,0.094440,0.081037,0.948274,10233.0
replicate,0.960481,0.059584,0.871291,0.987250,0.433814,0.343360,81.0
25_2.5_0.5,0.625322,1.059348,0.222343,0.094274,0.006780,0.990444,27.0
25_2.5_1,0.637027,1.092272,0.275464,0.113406,0.017683,0.979159,78.0
25_4_0.5,0.637027,1.145904,0.231005,0.090119,0.006942,0.994814,29.0
25_4_1,0.641538,1.175621,0.281169,0.147482,0.019032,0.984399,86.0
50_2.5_0.5,0.648196,1.090179,0.245028,0.098187,0.008877,0.990514,49.0
50_2.5_1,0.610932,1.116113,0.285429,0.088871,0.019356,0.984231,101.0
50_4_0.5,0.662908,1.225185,0.247837,0.073422,0.008875,0.993690,52.0
50_4_1,0.656680,1.252614,0.283532,0.107680,0.019305,0.990027,98.0


In [9]:
merged = pd.concat(
    {(line, kind): df
     for line, (scaled, raw) in results.items()
     for kind, df in (("scaled", scaled), ("raw", raw))},
    names=["line", "kind", "setting"],
)
merged.reset_index(inplace = True)

In [10]:
merged.sort_values('overall', ascending = False)

,line,kind,setting,pds,mse,fid,reach,jac,nmae,overall,n_sig
32,H1,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,81.0
1,HCT116,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
14,HCT116,scaled,100_2.5_2.5,0.253340,1.000000,0.038980,0.230816,0.006855,0.155218,0.280868,393.5
8,HCT116,scaled,50_4_0.5,0.334918,0.940205,0.119764,0.140670,0.017798,0.021866,0.262537,52.0
12,HCT116,scaled,100_4_0.5,0.224435,1.000000,0.092519,0.164484,0.013276,0.021592,0.252718,103.0
...,...,...,...,...,...,...,...,...,...,...,...
57,H1,raw,100_2.5_0.5,0.635524,1.125154,0.269827,0.095130,0.012994,0.994902,NaN,89.0
58,H1,raw,100_2.5_1,0.669566,1.144323,0.294650,0.111950,0.020540,0.980318,NaN,132.0
59,H1,raw,100_4_0.5,0.642504,1.336385,0.274099,0.100017,0.013701,1.010025,NaN,97.0
60,H1,raw,100_4_1,0.657539,1.357260,0.298409,0.106997,0.022094,0.998163,NaN,139.0


In [11]:
merged_no_lines = merged.copy().drop(columns = 'line')

In [12]:
merged_no_lines.groupby(['kind', 'setting']).mean().reset_index().sort_values(by = 'overall', ascending = False)

,kind,setting,pds,mse,fid,reach,jac,nmae,overall,n_sig
30,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,41.00
17,scaled,100_2.5_2.5,0.281171,0.500000,-0.171592,0.146024,-0.035925,0.055645,0.129220,555.75
19,scaled,100_4_1,0.296637,0.500000,-0.348170,0.045579,-0.077347,-0.013009,0.067282,131.50
27,scaled,50_4_1,0.224532,0.500000,-0.363203,0.092291,-0.080505,-0.004760,0.061392,85.75
26,scaled,50_4_0.5,0.344348,0.470102,-0.411870,0.058564,-0.093379,-0.026607,0.056860,52.00
18,scaled,100_4_0.5,0.266952,0.500000,-0.384559,0.085365,-0.088799,-0.040245,0.056452,100.00
23,scaled,25_4_1,0.221452,0.466544,-0.376701,0.093736,-0.079840,-0.004446,0.053458,68.00
16,scaled,100_2.5_1,0.278414,0.252931,-0.350048,0.095960,-0.079822,0.018067,0.035917,119.50
22,scaled,25_4_0.5,0.309437,0.386252,-0.421171,0.025774,-0.093926,-0.030946,0.029237,28.00
21,scaled,25_2.5_1,0.314319,0.167332,-0.360608,0.108945,-0.081883,0.016019,0.027354,63.50


In [13]:
merged[merged['kind'] == 'scaled']

,line,kind,setting,pds,mse,fid,reach,jac,nmae,overall,n_sig
0,HCT116,scaled,baseline,0.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,10297.0
1,HCT116,scaled,replicate,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
2,HCT116,scaled,25_2.5_0.5,0.315647,0.209666,0.152423,0.073310,0.021486,0.051395,0.137321,26.0
3,HCT116,scaled,25_2.5_1,0.331064,0.334664,0.136165,0.196648,0.015822,0.083094,0.182909,49.0
4,HCT116,scaled,25_4_0.5,0.321300,0.772504,0.153634,0.056388,0.022182,0.015046,0.223509,27.0
5,HCT116,scaled,25_4_1,0.135534,0.933089,0.086195,0.128062,0.016085,0.050829,0.224965,50.0
6,HCT116,scaled,50_2.5_0.5,0.220195,0.302585,0.136453,0.096686,0.017019,0.054505,0.137907,49.0
7,HCT116,scaled,50_2.5_1,0.251799,0.413995,0.122169,0.115991,0.014291,0.063227,0.163578,66.0
8,HCT116,scaled,50_4_0.5,0.334918,0.940205,0.119764,0.140670,0.017798,0.021866,0.262537,52.0
9,HCT116,scaled,50_4_1,0.108813,1.000000,0.105824,0.169754,0.013979,0.059503,0.242979,73.5
